# Mina Voice Studio — Kaggle GPU cartoon voice benchmark
**Purpose:** generate *real* Qwen3 VoiceDesign character seeds, reuse them with Qwen Base voice cloning, compare speaker embeddings across lines, and audition Qwen CustomVoice emotion control. No paid API keys, no proprietary voices.
**Important:** Kaggle accelerator must be enabled; enable internet if allowed for downloading public model weights. Free GPU allocation is quota-limited. Official Qwen3 does **not** list Arabic; this notebook tests English.
**Licenses:** Qwen3-TTS Apache 2.0. Use only original synthetic voices or authorized recordings. Outputs are for a local benchmark, not proof of ElevenLabs equivalence.
**Acoustic evidence:** includes generated WAV hashes, model/GPU versions, latency/RTF, speaker embeddings, clipping, WER/CER via real Faster-Whisper. All outputs are new measurements; this notebook must execute on a real GPU.

**2026-10-24 fix:** Kaggle's preinstalled PyAV can reject the `metadata_errors` argument used by Faster-Whisper's file decoder. ASR now decodes WAV via SoundFile, converts to 16 kHz with SoXR, and passes NumPy float32 directly to Faster-Whisper. Genuine output WAVs are archived before optional ASR starts, even if the ASR stage fails. This is a compatibility fix, not a synthetic success result.


In [ ]:
!python -m pip -q install qwen-tts soundfile resemblyzer faster-whisper jiwer soxr
import torch
assert torch.cuda.is_available(), "Enable GPU in Kaggle Notebook Settings → Accelerator; GPU is not active."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:",torch.__version__)


In [ ]:
import gc, json, zipfile, time, itertools
from pathlib import Path
import numpy as np
import soundfile as sf
from qwen_tts import Qwen3TTSModel
ROOT=Path("/kaggle/working/mina-cartoon-benchmark");ROOT.mkdir(parents=True,exist_ok=True)
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
def load(name):
    return Qwen3TTSModel.from_pretrained(name,device_map="cuda:0",dtype=DTYPE,attn_implementation="sdpa")
def release(model):
    del model;gc.collect();torch.cuda.empty_cache()
def save(name,wav,sr):
    p=ROOT/name;sf.write(str(p),np.asarray(wav).reshape(-1),int(sr),subtype="PCM_16");return p
def finite_clip(wav,sr):
    audio=np.asarray(wav)
    assert np.isfinite(audio).all() and audio.size>sr//4,"Bad/empty generated waveform"
print("Artifacts:",ROOT)


## Stage A — design two distinct synthetic voices from text (no reference recording)
Do not repeatedly resample until the metrics look flattering. Record all attempts, including failures.


In [ ]:
CHARACTERS=[
 {"name":"Shoko","description":"Young adult woman, bright American English, soft husky undertone, dry witty sarcastic edge, highly articulate and conversational.","seed":"Seriously? I survived university just to almost get hit by a car?"},
 {"name":"Robot","description":"Adult male synthetic announcer, light metallic resonant timbre, measured calm American English, low midrange pitch and crystal clear diction.","seed":"The laboratory doors are open. Please proceed to the demonstration."}
]
design=load("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
seeds={}
for char in CHARACTERS:
    start=time.monotonic()
    audio,sr=design.generate_voice_design(text=char["seed"],language="English",instruct=char["description"])
    finite_clip(audio[0],sr)
    char["path"]=save(char["name"]+"_seed.wav",audio[0],sr)
    seeds[char["name"]]=char["path"]
    print("Designed",char["name"],"seconds:",round(time.monotonic()-start,1))
del design;gc.collect();torch.cuda.empty_cache()


## Stage B — reuse an identical voice-clone prompt across lines
This is the key to keeping a character's voice consistent. The Qwen Base clone checkpoint has **no per-line emotion instruct parameter**; preset CustomVoice below provides that separately.


In [ ]:
LINES={
 "Shoko":["Oh, fantastic. Another assignment due yesterday!","Welcome to the show where everything goes wrong.","I just wanted five minutes of peace and quiet."],
 "Robot":["Warning. The coffee machine has become self-aware.","I recommend terminating the experiment immediately.","You may now initiate the emergency shutdown."]
}
base=load("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
clips=[]
for char in CHARACTERS:
    prompt=base.create_voice_clone_prompt(ref_audio=str(char["path"]),ref_text=char["seed"],x_vector_only_mode=False)
    for idx,line in enumerate(LINES[char["name"]],1):
        start=time.monotonic()
        wav,sr=base.generate_voice_clone(text=line,language="English",voice_clone_prompt=prompt)
        finite_clip(wav[0],sr)
        path=save(f"{char['name']}_{idx:02}.wav",wav[0],sr)
        clips.append({"character":char["name"],"line":idx,"text":line,"audio":str(path.name),"latency_s":round(time.monotonic()-start,2)})
        print(char["name"],idx,"in",clips[-1]["latency_s"],"sec")
del base;gc.collect();torch.cuda.empty_cache()


## Stage C — independent acting/emotion test on *preset* Qwen CustomVoice
The model explicitly accepts `instruct=` for preset voices; this is **not** the same as changing the cloned Shoko voice's emotion.


In [ ]:
custom=load("Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice")
for mood,instruct in [("sarcastic","Speak with dry, obvious sarcasm."),("angry","Speak with audible frustration and anger."),("crying","Speak tearfully with a trembling, sad voice.")]:
    audio,sr=custom.generate_custom_voice(text="Seriously? You cannot be serious right now!",language="English",speaker="Ryan",instruct=instruct)
    finite_clip(audio[0],sr)
    save("Ryan_"+mood+".wav",audio[0],sr)
del custom;gc.collect();torch.cuda.empty_cache()


## Stage D — independent speaker-embedding consistency report
Resemblyzer embeddings are only a **proxy**. Low similarity can reflect emotion, accent, pronunciation or noise rather than a true speaker change. Report all pairs; never treat a fixed threshold as certified identity.


In [ ]:
from resemblyzer import VoiceEncoder,preprocess_wav
encoder=VoiceEncoder(device="cpu")
vec={}
for item in clips:
    path=ROOT/item["audio"]
    vec[(item["character"],item["line"])]=encoder.embed_utterance(preprocess_wav(path))
def cos(a,b):return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
within=[];between=[];pairwise=[]
for a,b in itertools.combinations(clips,2):
    score=cos(vec[(a["character"],a["line"])],vec[(b["character"],b["line"])])
    same=a["character"]==b["character"]
    pairwise.append({"first":a["audio"],"second":b["audio"],"same_character":same,"cosine":round(score,4)})
    (within if same else between).append(score)
report={"gpu":torch.cuda.get_device_name(0),"precision":str(DTYPE),"models":["VoiceDesign-1.7B","Base-1.7B","CustomVoice-1.7B"],
    "within_character_mean":round(float(np.mean(within)),4),"within_character_min":round(min(within),4),
    "between_character_mean":round(float(np.mean(between)),4),"separation_margin":round(float(np.mean(within)-np.mean(between)),4),
    "pairs":pairwise,"clips":clips,"pass":None,
    "limitations":"English-biased speaker embeddings are a proxy. Blind human listening and further runs required."}
(ROOT/"metrics.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in report.items() if k not in ("pairs","clips")},indent=2))

# Preserve genuine generated WAVs and speaker statistics even if optional
# downstream ASR encounters a Kaggle system-library incompatibility.
ARCHIVE=Path("/kaggle/working/mina-cartoon-voices.zip")
with zipfile.ZipFile(ARCHIVE,"w",compression=zipfile.ZIP_DEFLATED) as pack:
    for p in ROOT.glob("*.wav"): pack.write(p,p.name)
    pack.write(ROOT/"metrics.json","metrics.json")
print("Preserved real Qwen WAVs and preliminary metrics:",ARCHIVE)


## Stage E — actual ASR word/character error and signal checks
This is an independent model-based intelligibility **proxy**, not ground-truth listener MOS. Every generated line is scored (no cherry-picking).


In [ ]:
from faster_whisper import WhisperModel
import soxr
from jiwer import wer, cer
from importlib.metadata import version as pkg_version
import hashlib, re, sys
asr_model=WhisperModel("base",device="cpu",compute_type="int8")
def normalized(text):
    return " ".join(re.sub(r"[^\w\s]"," ",text.lower()).split())
asr_rows=[]
technical_rows=[]
for row in clips:
    path=ROOT/row["audio"]
    audio,sr=sf.read(str(path),always_2d=False,dtype="float32")
    audio=np.asarray(audio)
    assert audio.size>0 and np.isfinite(audio).all()
    # Pass decoded 16 kHz NumPy PCM directly into Whisper. This bypasses
    # Kaggle's incompatible PyAV: av.open(..., metadata_errors="ignore").
    mono=np.mean(audio,axis=1) if audio.ndim==2 else audio
    audio16=soxr.resample(mono,int(sr),16000,quality="HQ") if int(sr)!=16000 else mono
    audio16=np.ascontiguousarray(audio16,dtype=np.float32)
    segments,info=asr_model.transcribe(audio16,language="en",beam_size=5,vad_filter=False)
    heard=" ".join(segment.text.strip() for segment in segments)
    expected=normalized(row["text"])
    decoded=normalized(heard)
    asr_rows.append({"audio":row["audio"],"reference":row["text"],
                     "transcribed":heard,"wer":round(float(wer(expected,decoded)),4),
                     "cer":round(float(cer(expected,decoded)),4)})
    technical_rows.append({
       "audio":row["audio"],"sample_rate":int(sr),
       "duration_s":round(audio.shape[0]/sr,3),
       "quiet_percent":round(float(np.mean(np.abs(audio)<=10**(-50/20)))*100,3),
       "clipped_percent":round(float(np.mean(np.abs(audio)>=.999))*100,3),
       "sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
       "latency_s":row["latency_s"],
       "rtf":round(row["latency_s"]/(audio.shape[0]/sr),3),
    })
report["asr"]={"model":"faster-whisper-base-CPU-int8","mean_wer":round(float(np.mean([x["wer"] for x in asr_rows])),4),
               "mean_cer":round(float(np.mean([x["cer"] for x in asr_rows])),4),
               "lines":asr_rows}
report["signal"] = technical_rows
report["environment"] = {"python":sys.version.split()[0],"torch":torch.__version__,
                         "qwen_tts":pkg_version("qwen-tts"),
                         "gpu_peak_torch_allocated_mib":round(torch.cuda.max_memory_allocated()/(1024**2),2)}
report["interpretation"]="WER/CER and speaker cosine are model-dependent proxies. Human blind listening is still required."
(ROOT/"metrics.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
print("Mean WER:",report["asr"]["mean_wer"],"CER:",report["asr"]["mean_cer"])


In [ ]:
ARCHIVE=Path("/kaggle/working/mina-cartoon-voices.zip")
with zipfile.ZipFile(ARCHIVE,"w",compression=zipfile.ZIP_DEFLATED) as pack:
    for p in ROOT.glob("*.wav"):pack.write(p,p.name)
    pack.write(ROOT/"metrics.json","metrics.json")
print("Download the actual generated audio and metrics from the Kaggle Notebook Output panel:",ARCHIVE)
